In [ ]:
#!pip install tqdm
#!pip install statsmodels

In [8]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from statsmodels.stats.multitest import multipletests

In [9]:
otu = pd.read_csv("IPS_out/IPS_concat.txt", sep = '\t')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu['count'] = 1
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="L",
    values="pa",
    fill_value=0
)

In [10]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)

results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

meta_IPS = pd.read_csv("IPS_metadata.txt", sep="\t")

results_df = pd.merge(results_df,meta_IPS, left_on = "otu", right_on = "ENTRY_AC", how = "left")

results_df.to_csv(f"stats_out/fisher_IPS_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████| 18623/18623 [01:07<00:00, 276.67it/s]


In [12]:
otu = pd.read_csv("IPS_out/IPS_concat.txt", sep = '\t')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu['count'] = 1
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'N']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="L",
    values="pa",
    fill_value=0
)

In [13]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)

results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

meta_IPS = pd.read_csv("IPS_metadata.txt", sep="\t")

results_df = pd.merge(results_df,meta_IPS, left_on = "otu", right_on = "ENTRY_AC", how = "left")

results_df.to_csv(f"stats_out/fisher_Nonprog_IPS_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████| 18015/18015 [00:30<00:00, 584.90it/s]


In [14]:
otu = pd.read_csv("IPS_out/IPS_concat.txt", sep = '\t')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu['count'] = 1
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Prog-Nonprog between 20-26y"] == 'P']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="L",
    values="pa",
    fill_value=0
)

In [15]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "20"
GroupB = "26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)

results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

meta_IPS = pd.read_csv("IPS_metadata.txt", sep="\t")

results_df = pd.merge(results_df,meta_IPS, left_on = "otu", right_on = "ENTRY_AC", how = "left")

results_df.to_csv(f"stats_out/fisher_Prog_IPS_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████| 17535/17535 [00:29<00:00, 588.95it/s]


In [16]:
otu = pd.read_csv("IPS_out/IPS_concat.txt", sep = '\t')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu['count'] = 1
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="L",
    values="pa",
    fill_value=0
)

In [17]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)

results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

meta_IPS = pd.read_csv("IPS_metadata.txt", sep="\t")

results_df = pd.merge(results_df,meta_IPS, left_on = "otu", right_on = "ENTRY_AC", how = "left")

results_df.to_csv(f"stats_out/fisher_IPS_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████| 18623/18623 [00:31<00:00, 586.45it/s]


In [18]:
otu = pd.read_csv("IPS_out/IPS_concat.txt", sep = '\t')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu['count'] = 1
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Biopsy_collection_date_year"] == '20']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="L",
    values="pa",
    fill_value=0
)

In [19]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)

results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

meta_IPS = pd.read_csv("IPS_metadata.txt", sep="\t")

results_df = pd.merge(results_df,meta_IPS, left_on = "otu", right_on = "ENTRY_AC", how = "left")

results_df.to_csv(f"stats_out/fisher_year_20_IPS_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|██████████████| 17661/17661 [00:31<00:00, 564.64it/s]


In [20]:
otu = pd.read_csv("IPS_out/IPS_concat.txt", sep = '\t')
otu = otu.rename(columns={'file': 'Metagenomic_file_name'})
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu['count'] = 1
otu

metadata = pd.read_csv("metadata.txt", sep = '\t', dtype=str)

dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')

dfm["pa"] = (dfm["count"] > 0).astype(int)

dfm = dfm[dfm["Biopsy_collection_date_year"] == '26']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="L",
    values="pa",
    fill_value=0
)

In [21]:
#filter df
GroupCol = "Prog-Nonprog between 20-26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_fisher(otu):
    A_vec = otu_A[otu].values
    B_vec = otu_B[otu].values

    # 2x2 contingency table values
    A_pos = A_vec.sum()
    A_neg = len(A_vec) - A_pos
    B_pos = B_vec.sum()
    B_neg = len(B_vec) - B_pos

    contingency = np.array([
        [A_pos, A_neg],
        [B_pos, B_neg]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {
        "otu": otu,
        "Odds_Ratio": odds_ratio,
        "P-value": p_value,
        f"{GroupA}_pos": A_pos,
        f"{GroupA}_neg": A_neg,
        f"{GroupB}_pos": B_pos,
        f"{GroupB}_neg": B_neg
    }


otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)

results_df["FDR_BH"] = multipletests(results_df["P-value"], method="fdr_bh")[1]

meta_IPS = pd.read_csv("IPS_metadata.txt", sep="\t")

results_df = pd.merge(results_df,meta_IPS, left_on = "otu", right_on = "ENTRY_AC", how = "left")

results_df.to_csv(f"stats_out/fisher_year_26_IPS_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running Fisher tests: 100%|█████████████| 17458/17458 [00:16<00:00, 1070.08it/s]
